---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Setup


## Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


## Libraries


In [ ]:
source(here::here("data-cleaning", "00b-packages.r"))


## R Scripts


In [ ]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


## Load Mapping Data


In [ ]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


## Function Definitions


In [ ]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    cat("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0(
      "✅ All validation checks passed for section ",
      section_id, ":\n"
    ))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0(
        "^chk_", section_id,
        "_\\d{2}_(.+)$"
      ), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

custom_setdiff <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}

custom_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    mismatch_msg <- paste0(
      "X: ", x, " Y: ", y
    )
    return(c(FALSE, mismatch_msg))
  }
}

custom_check_mappings <- function(result_list) {
  # Keep only the FALSE mappings
  filtered <- lapply(result_list, function(x) Filter(Negate(isTRUE), x))
  filtered <- Filter(length, filtered) # drop empty lists

  if (length(filtered) == 0) {
    return(c(TRUE, NULL))
  } else {
    ticker <- 0
    for (col in names(filtered)) {
      msg_lines <- if (ticker == 0) {
        c(paste0("\n $ ", col, ":"))
      } else {
        c(msg_lines, paste0(" $ ", col, ":"))
      }
      ticker <- ticker + 1
      for (raw_val in names(filtered[[col]])) {
        # Extract the actual wrongly mapped value from the original result_list
        wrong_val <- result_list[[col]][[raw_val]]
        msg_lines <- c(
          msg_lines,
          paste0('   $ "', raw_val, '": MAPPING FAILED')
        )
      }
    }
    return(c(FALSE, paste(msg_lines, collapse = "\n")))
  }
}


# Load Files (Lengthy)


## Load final .rds


In [ ]:
dt_clean_bq_subset <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))
dt_clean_full <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_a_is_covid", ".rds"
  )
))


## Load raw file


In [ ]:
# for (year_to_load in year_range) {
dt_raw <- fread(
  file = here(
    raw_claims_path,
    paste0(full_claims_prefix, year_to_load, file_type)
  ), colClasses = "character", header = TRUE,
  encoding = "Latin-1", na.strings = na_values
)

# Drop columns
cols_to_drop <- intersect(colnames(dt_raw), c(drop_cols, drop_cols_manual))
if (length(cols_to_drop) > 0) {
  dt_raw <- dt_raw[, (cols_to_drop) := NULL]
}

avail_cols <- colnames(dt_raw)

# Rename Columns
setnames(dt_raw,
  old = avail_cols[avail_cols %in% names(column_mappings)],
  new = unlist(column_mappings[
    avail_cols[
      avail_cols %in% names(column_mappings)
    ]
  ])
)

dt_raw[dt_raw == ""] <- NA_character_
dt_raw[, id_series := trimws(id_series)]

# dt <- dt_raw

# # Function to count the number of duplicate values in a vector
# count_duplicates <- function(column) {
#   # Create a frequency table of the column's values
#   freq_table <- table(column)
#   # Identify values that occur more than once (duplicates)
#   duplicate_values <- freq_table[freq_table > 1]
#   # Sum the counts of duplicate values (total number of duplicate instances)
#   sum(duplicate_values)
# }

# # Count duplicates for each column
# duplicate_counts <- sapply(dt, count_duplicates)

# # Count unique values for each column
# unique_counts <- sapply(dt, uniqueN)

# # Count non-missing entries (not NA) for each column
# non_missing_counts <- sapply(dt, function(x) sum(!is.na(x)))

# # Combine all results into a single summary data.table
# result <- data.table(
#   Column = names(dt),
#   Unique_Count = as.integer(unique_counts),
#   Duplicate_Count = as.integer(duplicate_counts),
#   NonMissing_Count = as.integer(non_missing_counts)
# )

# # Display the result in console
# print(result)

# # Save the result to a CSV file using the year as the filename prefix
# fwrite(result, paste0(year_to_load, ".csv"))
# }


# Data Verification Proper


### Test Batch 01: No Column Mismatches


In [ ]:
cols_expected <- bq_cols
cols_actual <- colnames(dt_clean_bq_subset)
cols_schema <-
  if (eclaims_batch == "2023_2024") {
    fromJSON(here(
      "data-cleaning/r_scripts_v2",
      "bq_schema_cleaning.json"
    ))$name
  } else if (eclaims_batch == "2025") {
    fromJSON(here(
      "data-cleaning/r_scripts_v2",
      "bq_schema_cleaning_2025.json"
    ))$name
  }
chk_01_01_cols_match_expected <- custom_setdiff(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- custom_setdiff(cols_schema, cols_actual)
test_checks(1)


### Test Batch 02: No Row Count Mismatches


In [ ]:
nrow_expected <- dt_raw[, .N]
nrow_actual_full <- nrow(dt_clean_bq_subset)
chk_02_01_nrows_match_full <- custom_setequal(
  nrow_expected, nrow_actual_full
)

# Function to check if MD5 hashes have changed, returning TRUE if no change
check_md5_changes <- function(year_to_load) {
  # Function to calculate and save MD5 hash for a given file
  calculate_md5 <- function(file_path) {
    md5sum <- digest::digest(file_path, algo = "md5", file = TRUE)
    return(md5sum)
  }
  hash_cache_dir <- here::here("data-cleaning/debug/cache/partial_md5")
  dir.create(hash_cache_dir, recursive = TRUE, showWarnings = FALSE)
  hash_file_path <- here::here(
    hash_cache_dir,
    paste0("md5_hashes_", year_to_load, ".rds")
  )

  # Generate new MD5 hashes for each part
  current_hashes <- sapply(1:split_parts, function(part) {
    part_file <- here::here(
      raw_claims_parts_path,
      paste0(
        full_claims_prefix, year_to_load, "_part_",
        sprintf("%02d", part), "_of_", split_parts, ".rds"
      )
    )
    calculate_md5(part_file)
  })

  # Check if saved hashes exist
  if (file.exists(hash_file_path)) {
    saved_hashes <- readRDS(hash_file_path)
    # Return TRUE if hashes match, indicating no changes
    if (identical(saved_hashes, current_hashes)) {
      cat(paste(
        "✅ No changes in partial files for eclaims year",
        year_to_load, "\n"
      ))
      return(TRUE)
    }
  }
  return(FALSE)
}
nrow_actual_partial <- nrow_partial <- 0
if (!check_md5_changes(year_to_load)) {
  for (loop_part in 1:split_parts) {
    nrow_partial <- nrow(read_appropriate_file(loop_part))
    nrow_actual_partial <- nrow_actual_partial + nrow_partial
  }
  chk_02_02_nrows_match_partial <- custom_setequal(
    nrow_expected, nrow_actual_partial
  )
} else {
  chk_02_02_nrows_match_partial <- c(TRUE, NULL)
}

test_checks(2)


### Test Batch 03: No Exponential Form ID's


In [ ]:
id_series_rows <- dt_clean_bq_subset[grepl("e", id_series), .(id_series)]
# TODO: add check for id_lhio
id_pin_rows <- dt_clean_bq_subset[grepl("e", id_pin), .(id_series, id_pin)]
id_hci_rows <- dt_clean_bq_subset[grepl("e", id_hci), .(id_series, id_hci)]
id_series_expo <-
  if (!is.null(id_series_rows)) {
    nrow(id_series_rows)
  } else {
    0
  }
id_pin_expo <-
  if (!is.null(id_pin_rows)) {
    nrow(id_pin_rows)
  } else {
    0
  }
id_hci_expo <-
  if (!is.null(id_hci_rows)) {
    nrow(id_hci_rows)
  } else {
    0
  }
chk_03_01_id_series_expo <-
  custom_setequal(id_series_expo, 0)
chk_03_02_id_pin_expo <-
  custom_setequal(id_pin_expo, 0)
chk_03_03_id_hci_expo <-
  custom_setequal(id_hci_expo, 0)
test_checks(3)


### Test Batch 04: No Unmapped Categoricals


In [ ]:
mapping_results <- list()
if (to_debug) {
  cat("==================================================\n")
  flush.console()
}
# Subfunction: check mapping for one column
process_column_mapping <- function(
    col_name, dt_raw, dt_clean_bq_subset,
    expected_mappings, to_debug = FALSE) {
  if (to_debug) {
    cat(col_name, "\n--------------------------------------------------\n")
    flush.console()
  }
  result <- list()
  raw_col <- dt_raw[[col_name]]
  clean_col <- dt_clean_bq_subset[[col_name]]
  unique_raw_vals <- unique(na.omit(raw_col))
  for (raw_val in unique_raw_vals) {
    if (to_debug) {
      cat(raw_val, "\n")
      flush.console()
    }
    id_subset <- dt_raw[raw_col == raw_val, id_series]
    clean_subset <- unlist(
      dt_clean_bq_subset[id_series %chin% id_subset, .SD, .SDcols = col_name],
      use.names = FALSE
    )
    unique_clean_vals <- unique(na.omit(clean_subset))
    result_flag <-
      all(is.na(unique_clean_vals)) | length(unique_clean_vals) == 1
    result[[raw_val]] <- result_flag
    if (to_debug) {
      cat(str(id_subset), str(clean_subset), str(unique_clean_vals), sep = "\n")
      cat(result_flag, "\n--------------------------------------------------\n")
      flush.console()
    }
  }
  return(result)
}

mapping_results <- list()
if (to_debug) cat("==================================================\n")
if (to_debug) flush.console()
for (col_name in names(expected_mappings)) {
  mapping_results[[col_name]] <-
    process_column_mapping(
      col_name, dt_raw, dt_clean_bq_subset,
      expected_mappings, to_debug
    )
  if (to_debug) cat("==================================================\n")
  if (to_debug) flush.console()
}

if (to_debug) {
  str(mapping_results)
}

chk_04_01_mapping_results <- custom_check_mappings(mapping_results)
test_checks(04)


## Test Batch 05: No Unacceptable PDx


In [ ]:
dt_unacceptable_pdx <- dt_clean_bq_subset[
  !is.na(clin_pdx) & !clin_pdx %chin% acc_pdx,
  .(id_series, clin_pdx)
]
chk_05_01_no_unacceptable_pdx <-
  if (nrow(dt_unacceptable_pdx) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_unacceptable_pdx)), collapse = "\n"))
  }
test_checks(5)


## Test Batch 06: No Invalid Ages


In [ ]:
dt_invalid_ages_01 <- dt_clean_full[
  (!is.na(pat_bdate) & !is.na(date_adm) & !is.na(pat_age)) &
    pat_age != floor(as.numeric(as.Date(date_adm) - pat_bdate) / 365.25),
  .(id_series, date_adm, pat_bdate, pat_age)
]
dt_invalid_ages_02 <- dt_clean_full[
  (!is.na(pat_bdate) & !is.na(date_adm) & !is.na(pat_age)) &
    (pat_age < 0 | pat_age > 124),
  .(id_series, date_adm, pat_bdate, pat_age)
]
dt_invalid_age <- rbind(dt_invalid_ages_01, dt_invalid_ages_02)
chk_06_01_no_invalid_ages <-
  if (nrow(dt_invalid_age) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_age)), collapse = "\n"))
  }
test_checks(6)


## Test Batch 07: No Invalid Sex or Discharge


In [ ]:
dt_invalid_sex <- dt_clean_bq_subset[
  is.na(pat_sex) | !(pat_sex %in% c("M", "F")),
  .(id_series, pat_sex)
]
dt_invalid_discharge <- dt_clean_bq_subset[
  !(clin_discharge %in% c(NA_character_, 1, 2, 3, 4, 9)),
  .(id_series, clin_discharge)
]
chk_07_01_no_invalid_sex <-
  if (nrow(dt_invalid_sex) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_sex)), collapse = "\n"))
  }
chk_07_02_no_invalid_discharge <-
  if (nrow(dt_invalid_discharge) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_discharge)), collapse = "\n"))
  }
test_checks(7)


## Test Batch 08: No Date Before 1900-01-01


In [ ]:
dt_invalid_date_adm <- dt_clean_full[
  !is.na(date_adm) &
    date_adm < as.Date("1900-01-01"),
  .(id_series, date_adm)
]
dt_invalid_date_dis <- dt_clean_full[
  !is.na(date_dis) &
    date_dis < as.Date("1900-01-01"),
  .(id_series, date_dis)
]
dt_invalid_date_rec <- dt_clean_full[
  !is.na(date_rec) &
    date_rec < as.Date("1900-01-01"),
  .(id_series, date_rec)
]
dt_invalid_date_ref <- dt_clean_full[
  !is.na(date_ref) &
    date_ref < as.Date("1900-01-01"),
  .(id_series, date_ref)
]
dt_invalid_date_check <- dt_clean_full[
  !is.na(date_check) &
    date_check < as.Date("1900-01-01"),
  .(id_series, date_check)
]
dt_invalid_pat_bdate <- dt_clean_full[
  !is.na(pat_bdate) &
    pat_bdate < as.Date("1900-01-01"),
  .(id_series, pat_bdate)
]
dt_invalid_date_ext <- dt_clean_full[
  !is.na(date_ext) &
    date_ext < as.Date("1900-01-01"),
  .(id_series, date_ext)
]

chk_08_01_no_invalid_date_adm <-
  if (nrow(dt_invalid_date_adm) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_adm)), collapse = "\n"))
  }

chk_08_02_no_invalid_date_dis <-
  if (nrow(dt_invalid_date_dis) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_dis)), collapse = "\n"))
  }

chk_08_03_no_invalid_date_rec <-
  if (nrow(dt_invalid_date_rec) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_rec)), collapse = "\n"))
  }

chk_08_04_no_invalid_date_ref <-
  if (nrow(dt_invalid_date_ref) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_ref)), collapse = "\n"))
  }

chk_08_05_no_invalid_date_check <-
  if (nrow(dt_invalid_date_check) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_check)), collapse = "\n"))
  }

chk_08_06_no_invalid_pat_bdate <-
  if (nrow(dt_invalid_pat_bdate) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_pat_bdate)), collapse = "\n"))
  }

chk_08_07_no_invalid_date_ext <-
  if (nrow(dt_invalid_date_ext) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_invalid_date_ext)), collapse = "\n"))
  }
test_checks(8)


## Test Batch 09: Generate Mappings

(Only works right after a data-cleaning run)


In [ ]:
# Load parameter file
source("~/pids-drg-claims/data-cleaning/00a-parameters.r")

# Loop through each year in the specified range
for (year_to_process in year_range) {
  print(year_to_process)

  # Loop over both sampled and full versions
  for (sample_flag in c(TRUE, FALSE)) {
    mapping_type <- if (sample_flag) "sampled" else "full"
    list_of_divisors <- if (sample_flag) c(625, 125, 25, 5) else c(1)

    # Loop over each divisor
    for (sample_size_divisor in list_of_divisors) {
      # Construct suffix based on sampling flag
      suffix <- paste0(ifelse(exists("sample_flag") && sample_flag,
        paste0("_sampled_", sample_size_divisor, "_"), "_full_"
      ))

      # Validate the suffix format
      valid_suffix_pattern <- "_full_|_sampled_\\d+_"
      if (!grepl(valid_suffix_pattern, suffix)) {
        stop("Invalid suffix: ", suffix, ". Expected '_full_' or '_sampled_<sample_size_divisor>_'")
      }

      # Get matching filenames from the checkpoint folder
      files <- list.files(
        path = paste0(
          "~/pids-drg-claims/data-cleaning/data/chkpts/",
          eclaims_batch, "/chkpt_12_mapping/"
        ),
        pattern = paste0("map_", year_to_process, suffix, ".*fork_\\d+_part_\\d+\\.rds"),
        full.names = TRUE
      )

      # Skip if no files found
      if (length(files) == 0) {
        message("No files found for the specified year and suffix: ", year_to_process, " ", suffix)
        message("Check: ", here(paste0(
          "~/pids-drg-claims/data-cleaning/data/chkpts/", eclaims_batch,
          "/chkpt_12_mapping/", mapping_type, "_final_mappings"
        )))
        next
      }

      # Function to parse timestamp from filenames
      parse_filename <- function(filename) {
        pattern <- paste0("map_(\\d{4})", suffix, "(\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\.\\d+)_fork_\\d+_part_\\d+\\.rds")
        matches <- stringr::str_match(filename, pattern)

        if (!is.na(matches[1])) {
          return(list(
            year_to_load = as.integer(matches[2]),
            timestamp = matches[3],
            filename = filename
          ))
        } else {
          return(NULL)
        }
      }

      # Apply parser to all files
      parsed_files <- lapply(files, parse_filename)
      parsed_files <- Filter(Negate(is.null), parsed_files)

      if (length(parsed_files) == 0) {
        message("No valid parsed files found for: ", year_to_process, " and suffix: ", suffix)
        next
      }

      # Use latest timestamp and convert to Unix time
      latest_timestamp <- max(sapply(parsed_files, `[[`, "timestamp"))
      latest_timestamp_unix <- as.integer(as.POSIXct(latest_timestamp, format = "%Y-%m-%d %H:%M:%OS"))

      # Filter for files matching the latest timestamp
      latest_files <- Filter(function(x) {
        x$timestamp == latest_timestamp
      }, parsed_files)

      # Construct output folder paths using numeric timestamp
      partial_output_folder <- paste0(
        "~/pids-drg-claims/data-cleaning/data/chkpts/", eclaims_batch,
        "/chkpt_12_mapping/", mapping_type, "_rds_files/", year_to_process,
        suffix, latest_timestamp_unix, "/"
      )

      final_output_folder <- paste0(
        "~/pids-drg-claims/data-cleaning/data/chkpts/", eclaims_batch,
        "/chkpt_12_mapping/", mapping_type, "_final_mappings/", year_to_process,
        suffix, latest_timestamp_unix, "/"
      )

      dir.create(final_output_folder, recursive = TRUE, showWarnings = FALSE)
      dir.create(partial_output_folder, recursive = TRUE, showWarnings = FALSE)

      # Read and bind ICD mappings
      final_icd_dt <- data.table::rbindlist(lapply(latest_files, function(file_info) {
        mappings_list <- readRDS(file_info$filename)
        mappings_list$icd_mappings
      }), use.names = TRUE, fill = TRUE)

      # Read and bind RVS mappings
      final_rvs_dt <- data.table::rbindlist(lapply(latest_files, function(file_info) {
        mappings_list <- readRDS(file_info$filename)
        mappings_list$rvs_mappings
      }), use.names = TRUE, fill = TRUE)

      # Deduplicate mappings while keeping the most frequent mapping
      final_icd_dt <- final_icd_dt[,
        .(mapped_code = mapped_code[1], count = .N),
        by = raw_code
      ][order(-count)]

      final_rvs_dt <- final_rvs_dt[,
        .(mapped_code = mapped_code[1], count = .N),
        by = raw_code
      ][order(-count)]

      # Stats function to assess unmapped codes
      calc_unmappable_stats <- function(dt, code_type) {
        total_count <- sum(dt$count)
        unmappable_count <- sum(dt[mapped_code == "_"]$count)
        unmappable_percent <- sprintf("%.2f%%", (unmappable_count / total_count) * 100)

        unique_total <- nrow(dt)
        unique_unmappable_count <- nrow(dt[mapped_code == "_"])
        unique_unmappable_percent <- sprintf("%.2f%%", (unique_unmappable_count / unique_total) * 100)

        return(data.table::data.table(
          code_type = code_type,
          unmappable_count = unmappable_count,
          total_count = total_count,
          unmappable_percent = unmappable_percent,
          unique_unmappable_count = unique_unmappable_count,
          unique_total = unique_total,
          unique_unmappable_percent = unique_unmappable_percent
        ))
      }

      # Generate and save summary statistics
      icd_stats <- calc_unmappable_stats(final_icd_dt, "icd")
      rvs_stats <- calc_unmappable_stats(final_rvs_dt, "rvs")
      summary_stats <- rbind(icd_stats, rvs_stats)

      output_summary_csv <- paste0(
        final_output_folder, "summary_stats_",
        year_to_process, suffix, latest_timestamp_unix, ".csv"
      )
      data.table::fwrite(summary_stats, output_summary_csv)

      # Save final RDS list
      final_mappings_list <- list(
        final_icd_dt = final_icd_dt,
        final_rvs_dt = final_rvs_dt
      )
      output_rds <- paste0(
        final_output_folder, "final_map_",
        year_to_process, suffix, latest_timestamp_unix, ".rds"
      )
      saveRDS(final_mappings_list, output_rds)

      # Save debug CSVs
      output_icd_csv <- paste0(
        final_output_folder, "icd_",
        year_to_process, suffix, latest_timestamp_unix, ".csv"
      )
      output_rvs_csv <- paste0(
        final_output_folder, "rvs_",
        year_to_process, suffix, latest_timestamp_unix, ".csv"
      )

      data.table::fwrite(final_icd_dt, output_icd_csv)
      data.table::fwrite(final_rvs_dt, output_rvs_csv)

      # Move the original partial .rds files to the organized folder
      file.rename(files, file.path(partial_output_folder, basename(files)))

      # Print confirmation messages
      message("Final mappings saved to: ", output_rds)
      message("ICD mappings saved to: ", output_icd_csv)
      message("RVS mappings saved to: ", output_rvs_csv)
      message("Partial mapping files moved to: ", partial_output_folder)
    }
  }
}
